In [ ]:
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf
from MEDS_transforms.runner import main

config_dir = "/workspaces/.venv/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs"

with initialize_config_dir(config_dir=config_dir, version_base=None):
    cfg = compose(
        config_name="runner",
        overrides=[
            "pipeline_config_fp=/workspaces/.venv/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs/ETL.yaml",
            "do_overwrite=True",
            "do_copy=False",
            "~parallelize",
            "hydra.searchpath=[pkg://MEDS_transforms.configs]",
        ],
    )

print(OmegaConf.to_yaml(cfg))

In [1]:
from pathlib import Path
from omegaconf import OmegaConf
from MEDS_transforms.runner import load_yaml_file, run_stage, RESERVED_CONFIG_NAMES

cfg = OmegaConf.create({
    "pipeline_config_fp": "/workspaces/.venv/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs/ETL.yaml",
    "log_dir": "/workspaces/OpenICU.example/output/project/MEDS_extract-MIMIC_IV-root_output_dir/logs",
    "do_profile": False,
    "_stage_runners": {},
})

pipeline_config_fp = Path(cfg.pipeline_config_fp)
if pipeline_config_fp.stem in RESERVED_CONFIG_NAMES:
    raise ValueError(f"Bad pipeline config name: {pipeline_config_fp}")

pipeline_config = load_yaml_file(cfg.pipeline_config_fp)
stages = pipeline_config.get("stages", [])
cfg._local_pipeline_config = pipeline_config

if "parallelize" in cfg._stage_runners:
    default_parallelization_cfg = cfg._stage_runners.parallelize
elif "parallelize" in cfg:
    default_parallelization_cfg = cfg.parallelize
else:
    default_parallelization_cfg = None

for stage in stages:
    print("RUN STAGE:", stage)
    run_stage(cfg, stage, default_parallelization_cfg=default_parallelization_cfg)

RUN STAGE: shard_events


ValueError: Stage shard_events failed via MEDS_extract-shard_events --config-dir=/workspaces/.venv/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs --config-name=ETL 'hydra.searchpath=[pkg://MEDS_transforms.configs]' stage=shard_events with return code 1.

In [3]:
import subprocess
from MEDS_transforms.runner import run_stage

def debug_runner(cmd, shell, capture_output):
    print("=== COMMAND ===")
    print(cmd)
    print()

    result = subprocess.run(cmd, shell=shell, capture_output=True, text=True)

    print("=== STDOUT ===")
    print(result.stdout)
    print()

    print("=== STDERR ===")
    print(result.stderr)
    print()

    print("=== RETURNCODE ===")
    print(result.returncode)
    print()

    return result

stage = "shard_events"
run_stage(cfg, stage, default_parallelization_cfg=default_parallelization_cfg, runner_fn=debug_runner)

=== COMMAND ===
MEDS_extract-shard_events --config-dir=/workspaces/.venv/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs --config-name=ETL 'hydra.searchpath=[pkg://MEDS_transforms.configs]' stage=shard_events

=== STDOUT ===


=== STDERR ===
An error occurred during Hydra's exception formatting:
AssertionError()
Traceback (most recent call last):
  File "/workspaces/.venv/bin/MEDS_extract-shard_events", line 6, in <module>
    sys.exit(main())
             ~~~~^^
  File "/workspaces/.venv/lib/python3.13/site-packages/hydra/main.py", line 94, in decorated_main
    _run_hydra(
    ~~~~~~~~~~^
        args=args,
        ^^^^^^^^^^
    ...<3 lines>...
        config_name=config_name,
        ^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/workspaces/.venv/lib/python3.13/site-packages/hydra/_internal/utils.py", line 394, in _run_hydra
    _run_app(
    ~~~~~~~~^
        run=args.run,
        ^^^^^^^^^^^^^
    ...<5 lines>...
        overrides=overrides,
        ^^^^^^^^^^^^^^^^^^^^
    )
 

AttributeError: 'str' object has no attribute 'decode'